In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

sns.set_theme(style="whitegrid")

In [ ]:
data_path = Path('Customer Churn.csv')

if not data_path.exists():
    try:
        import kagglehub
        from shutil import copy2

        download_path = Path(kagglehub.dataset_download('muhammadshahidazeem/customer-churn-dataset'))
        # Prefer the training file if present
        candidates = sorted(download_path.rglob('*.csv'))
        preferred = [p for p in candidates if 'training' in p.name.lower()]
        src = preferred[0] if preferred else (candidates[0] if candidates else None)

        if src is None:
            print('Kaggle download succeeded but no CSV files were found.')
        else:
            copy2(src, data_path)
            print(f'Downloaded dataset to: {data_path}')
    except ModuleNotFoundError:
        print('Missing Customer Churn.csv and kagglehub is not installed.')
        print('Install it with: pip install kagglehub')
    except Exception as e:
        print('Could not download dataset from Kaggle:', e)
        print('If needed, set up Kaggle credentials (~/.kaggle/kaggle.json).')

if data_path.exists():
    df = pd.read_csv(data_path)

    # Normalize column names to snake_case
    df = df.rename(columns={c: c.strip().lower().replace(' ', '_') for c in df.columns})

    # Map Kaggle schema to the notebook’s expected names
    rename_map = {}
    if 'customerid' in df.columns and 'customer_id' not in df.columns:
        rename_map['customerid'] = 'customer_id'
    if 'churn' in df.columns and 'churned' not in df.columns:
        rename_map['churn'] = 'churned'
    if 'tenure' in df.columns and 'tenure_months' not in df.columns:
        rename_map['tenure'] = 'tenure_months'

    df = df.rename(columns=rename_map)

    # Ensure churned is numeric 0/1
    if 'churned' in df.columns:
        df['churned'] = pd.to_numeric(df['churned'], errors='coerce').fillna(0).astype(int)

    HAS_DATA = True
else:
    HAS_DATA = False
    df = pd.DataFrame()
    print(f'Missing {data_path}. Add the dataset to this folder to run the full analysis.')

df.head() if HAS_DATA else df

FileNotFoundError: Missing customer_churn.csv. Place the dataset in this folder and re-run.

In [ ]:
if not HAS_DATA:
    print('Skipping df.info() (no dataset).')
else:
    df.info()

In [ ]:
if not HAS_DATA:
    print('Skipping churn EDA (no dataset).')
else:
    churn_rate = df['churned'].mean()
    print(f"Churn rate: {churn_rate:.2%}")

    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='churned')
    plt.title('Churned vs Not Churned')
    plt.show()

In [ ]:
if not HAS_DATA:
    print('Skipping tenure/feature EDA (no dataset).')
else:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x='churned', y='tenure_months')
    plt.title('Tenure vs Churn')
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x='churned', y='usage_frequency')
    plt.title('Usage Frequency vs Churn')
    plt.show()

## Modeling
We predict `churned` from a mix of numeric + categorical features using a standard sklearn preprocessing pipeline.

In [ ]:
if not HAS_DATA:
    print('Skipping modeling setup (no dataset).')
else:
    target = 'churned'
    drop_cols = ['customer_id']

    X = df.drop(columns=[target] + drop_cols)
    y = df[target]

    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('scaler', StandardScaler())]), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ],
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

In [ ]:
if not HAS_DATA:
    print('Skipping Logistic Regression (no dataset).')
else:
    log_reg = Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('model', LogisticRegression(max_iter=2000, class_weight='balanced')),
        ]
    )

    log_reg.fit(X_train, y_train)
    proba_lr = log_reg.predict_proba(X_test)[:, 1]
    pred_lr = (proba_lr >= 0.5).astype(int)

    print('Logistic Regression ROC AUC:', roc_auc_score(y_test, proba_lr))
    print(classification_report(y_test, pred_lr))

In [ ]:
if not HAS_DATA:
    print('Skipping Random Forest (no dataset).')
else:
    rf = Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('model', RandomForestClassifier(n_estimators=400, random_state=42, class_weight='balanced_subsample')),
        ]
    )

    rf.fit(X_train, y_train)
    proba_rf = rf.predict_proba(X_test)[:, 1]
    pred_rf = (proba_rf >= 0.5).astype(int)

    print('Random Forest ROC AUC:', roc_auc_score(y_test, proba_rf))
    print(classification_report(y_test, pred_rf))

In [ ]:
if not HAS_DATA:
    print('Skipping ROC curves (no dataset).')
else:
    fpr_lr, tpr_lr, _ = roc_curve(y_test, proba_lr)
    fpr_rf, tpr_rf, _ = roc_curve(y_test, proba_rf)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr_lr, tpr_lr, label='LogReg')
    plt.plot(fpr_rf, tpr_rf, label='RandomForest')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()
    plt.show()